In [ ]:
!pip install datasets

In [ ]:
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration, T5Tokenizer
import numpy as np
import datasets
from datasets import DatasetDict
from sklearn.model_selection import train_test_split
import random

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt


In [ ]:
# fix seeds for reproducibility
#torch.manual_seed(0)
#random.seed(0)
#np.random.seed(0)

In [ ]:
model_name = 'jbochi/madlad400-3b-mt'
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.43M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.6M [00:00<?, ?B/s]

In [ ]:
print(f"> Number of parameters in the model: {sum(p.numel() for p in model.parameters()):,}")


> Number of parameters in the model: 2,940,374,016


In [ ]:
gatitos: DatasetDict = datasets.load_dataset("google/smol", "gatitos__yue_zh") # type: ignore
smolsent: DatasetDict = datasets.load_dataset("google/smol", "smolsent__en_yue") # type: ignore
smoldoc: DatasetDict = datasets.load_dataset("google/smol", "smoldoc__en_yue") # type: ignore

README.md: 0.00B [00:00, ?B/s]

yue_zh.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

en_yue.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

en_yue.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


Define Hyperparameters

In [ ]:
epochs = 2
batch_size = 8
max_len = 128
learning_rate = 1e-4

Create dataloaders

In [ ]:
class SmolsentDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return self.dataset.__len__()

    def __getitem__(self, idx):
        return self.dataset[idx]["trg"], self.dataset[idx]["src"]

smolsent_train_validation = smolsent["train"].train_test_split(train_size=0.9, shuffle=True, seed=2)
smolsent_train_test = smolsent_train_validation["train"].train_test_split(train_size=8/9, shuffle=True, seed=2)

smolsent_train = smolsent_train_test["train"]
smolsent_validation = smolsent_train_validation["test"]
smolsent_test = smolsent_train_test["test"]

smolsent_validation_dataset = SmolsentDataset(smolsent_validation)
smolsent_test_dataset = SmolsentDataset(smolsent_test)
smolsent_train_dataset = SmolsentDataset(smolsent_train)

smolsent_train_dataloader = DataLoader(SmolsentDataset(smolsent_train), batch_size=batch_size, shuffle=True)
smolsent_validation_dataloader = DataLoader(SmolsentDataset(smolsent_validation), batch_size=batch_size, shuffle=True)
smolsent_test_dataloader = DataLoader(SmolsentDataset(smolsent_test), batch_size=batch_size, shuffle=True)


In [ ]:
from functools import reduce
from operator import add
canto = reduce(add, smoldoc["train"]["trgs"], [])
labels = reduce(add, smoldoc["train"]["srcs"], [])

canto_tokenized = tokenizer(canto, add_special_tokens=True)
labels_tokenized = tokenizer(labels, add_special_tokens=True)

# remove to avoid going over max length
filtered_pairs = [
    (c, l, c_tok, l_tok)
    for c, l, c_tok, l_tok in zip(
        canto,
        labels,
        canto_tokenized['input_ids'],
        labels_tokenized['input_ids']
    )
    if len(c_tok) <= 120 and len(l_tok) <= 120
]

canto_filtered = [pair[0] for pair in filtered_pairs]
labels_filtered = [pair[1] for pair in filtered_pairs]
canto_ids_filtered = [pair[2] for pair in filtered_pairs]
labels_ids_filtered = [pair[3] for pair in filtered_pairs]

from datasets import Dataset, DatasetDict, concatenate_datasets

filtered_dataset = Dataset.from_dict({
    "trg": canto_filtered,
    "src": labels_filtered
})

filtered_train_validation = filtered_dataset.train_test_split(train_size=0.9, shuffle=True, seed=2)
filtered_train_test = filtered_train_validation["train"].train_test_split(train_size=8/9, shuffle=True, seed=2)

filtered_train = filtered_train_test["train"]
filtered_validation = filtered_train_validation["test"]
filtered_test = filtered_train_test["test"]

smolsent_train = concatenate_datasets([smolsent_train, filtered_train])
smolsent_validation = concatenate_datasets([smolsent_validation, filtered_validation])
smolsent_test = concatenate_datasets([smolsent_test, filtered_test])

def preprocess(batch):
  trg = [f"<2en> {t}" for t in batch["trg"]]

  model_inputs = tokenizer(
      trg, padding="max_length", truncation=True, max_length=max_len
  )
  labels = tokenizer(
      batch["src"], padding="max_length", truncation=True, max_length=max_len
  )

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

smolsent_train_dataset = smolsent_train.map(preprocess, batched=True)
smolsent_test_dataset = smolsent_test.map(preprocess, batched=True)
smolsent_validation_dataset = smolsent_validation.map(preprocess, batched=True)

smolsent_train_dataloader = DataLoader(smolsent_train_dataset, batch_size=batch_size, shuffle=True)
smolsent_validation_dataloader = DataLoader(smolsent_validation_dataset, batch_size=batch_size, shuffle=True)
smolsent_test_dataloader = DataLoader(smolsent_test_dataset, batch_size=batch_size, shuffle=True)

print(f"Train size: {len(smolsent_train)}")
print(f"Validation size: {len(smolsent_validation)}")
print(f"Test size: {len(smolsent_test)}")

Map:   0%|          | 0/6265 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Train size: 6265
Validation size: 784
Test size: 784


In [ ]:
[x for x in smolsent_train['src'] if "Rohan" in x]

[]

In [ ]:
smolsent_train_dataset

Dataset({
    features: ['sl', 'tl', 'id', 'src', 'trg', 'is_src_orig', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 6265
})

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=epochs,
    report_to=[],
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_strategy='epoch',
    warmup_steps=100,
)


In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = smolsent_train_dataset,
    eval_dataset = smolsent_validation_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print('Start training')
trainer.train()
print('End training')

/tmp/ipython-input-278756648.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Start training


Step,Training Loss
500,0.039400
1000,0.024200
1500,0.027800


End training


In [ ]:
smolsent_validation_dataset[0]['src']

'Here, too, males’ education outcomes improved significantly.'

In [ ]:
text = "<2en> 羅漢係我兄弟，佢成日喺正午放風箏，就算天陰陰。"
input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
torch.manual_seed(42)
import random
random.seed(42)
out = model.generate(input_ids)
tokenizer.decode(out[0], skip_special_tokens=True)

'Roxanne is my sister, and she always kites at noon, even in the darkest'

In [ ]:
[x for x in smolsent_train['src'] if "Rohan" in x]

[]

In [ ]:
import torch
import shutil

torch.save(model.state_dict(), "model1212_epoch4.pt")



In [ ]:
# Source - https://stackoverflow.com/a/59371099
# Posted by codefun, modified by community. See post 'Timeline' for change history
# Retrieved 2025-12-08, License - CC BY-SA 4.0

from google.colab import drive
drive.mount('/content/gdrive',force_remount=True)


Mounted at /content/gdrive


In [ ]:
import shutil

shutil.copy("model1212_epoch4.pt", "/content/gdrive/MyDrive/model1212_epoch4.pt")

#!cp model.pt '/content/gdrive/'
#!ls -lt '/content/gdrive/My Drive/downloads/'


'/content/gdrive/MyDrive/model1212_epoch4.pt'

In [ ]:
!ls -lh /content/gdrive/MyDrive/


total 13G
-rw------- 1 root root   174 Dec  6  2020 '1. b3.gsheet'
-rw------- 1 root root  5.4M Jun  3  2017  20170507_102110.jpg
-rw------- 1 root root  8.8M Jun  3  2017  20170603_173654.jpg
-rw------- 1 root root  7.2M Jul 12  2017 '20170620_211226(0).jpg'
-rw------- 1 root root  1.1M Jul 12  2017  20170708_203430.jpg
-rw------- 1 root root  1.1M Jul 12  2017  20170708_223114.jpg
-rw------- 1 root root  922K Jul 12  2017  20170708_223200.jpg
-rw------- 1 root root   18M Sep 20  2017  20170917_125436_1.mp4
-rw------- 1 root root  4.2M Dec  4  2017  20171204_133833.jpg
-rw------- 1 root root  1.5M Mar  7  2018  20180306_210813.jpg
drwx------ 2 root root  4.0K Dec 15  2021  AASVK21
-rw------- 1 root root   311 Sep 15  2021  AASVK_PracticePoints.cup
-rw------- 1 root root   174 Dec 21  2020  A.gsheet
-rw------- 1 root root   174 Aug 28  2024 'Ansttelseskontrakt - Lasse Wendtland Edslev .gdoc'
-rw------- 1 root root   55K Aug 28  2024 'Ansttelseskontrakt - Lasse Wendtland Edslev .pdf'
-r

In [ ]:

from google.colab import files
files.download("model1212.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>